# Python 01 — Variables and Types (Production Mindset)

**Roadmap position:** Python → Core → Variables and types.

This notebook keeps the first roadmap topic small, but uses the habits expected in real software: precise names, type hints, validation at system boundaries, useful errors, and executable checks. You will still write and debug the key code yourself.

## Outcome

You can represent basic data safely, identify types, convert untrusted string input deliberately, and explain type failures as you would in an interview.

## Production conventions used today

- Names describe what a value means: `weekly_study_hours`, not `x`.
- Type hints document the contract; Python does not enforce them at runtime.
- Validate data at boundaries such as APIs, files, and user input.
- Raise errors that tell the caller what is wrong and how to fix it.
- Test the expected behaviour and important invalid inputs.

In [ ]:
# Read and run. These values are typed in source code, so they are already trusted.
candidate_name: str = 'Asha'
applications_submitted: int = 12
weekly_study_hours: float = 14.5
has_github_profile: bool = True
target_company: str | None = None

profile_values = {
    'candidate_name': candidate_name,
    'applications_submitted': applications_submitted,
    'weekly_study_hours': weekly_study_hours,
    'has_github_profile': has_github_profile,
    'target_company': target_company,
}

for field_name, field_value in profile_values.items():
    print(f'{field_name}: {field_value!r} ({type(field_value).__name__})')

## Your turn 1 — define a clear data contract

Create five appropriately named, type-hinted variables for your own profile: name (`str`), year of study (`int`), average daily study hours (`float`), GitHub availability (`bool`), and preferred company (`str | None`). Print them in a readable summary.

**Interview prompt:** Python is dynamically typed. What useful job do the type hints still perform?

In [4]:
# YOUR TURN — use production-quality names and type hints.
name : str = "Vatsal"
YOE : int = 2028
ADSH : float =8
git : bool =True
preferred_company : str | None = "JP Morgam"
# type hints help us improve code readaability and help linters and type checkers

## Boundary data is not trusted

HTTP request bodies, environment variables, CSV fields, and CLI input normally start as strings or unknown values. Do not let their representation leak into the rest of your program. Parse and validate them once at the boundary.

Notice that `bool('false')` is `True`: it asks whether the string is non-empty, not whether it represents a boolean.

In [12]:
from collections.abc import Mapping


def parse_boolean(raw_value: str) -> bool:
    """Convert a supported API boolean representation to a bool."""
    normalized_value = raw_value.strip().lower()
    if normalized_value == 'true':
        return True
    if normalized_value == 'false':
        return False
    raise ValueError(f"Expected 'true' or 'false', received {raw_value!r}.")


print(parse_boolean(' TRUE '))
print(parse_boolean('false'))
# Uncomment to inspect a useful failure:
# parse_boolean('yes')

True
False


## Debugging drill — fix the boundary conversion

This code joins a portal count from an API to an internal count. It should print `18`, but it fails.

1. Run it unchanged and read the full traceback.
2. Identify the type and source of each failing operand.
3. Fix the data at the boundary—not with a hidden workaround at every use site.
4. Add one assertion that would catch a bad result.

Do not merely change the final expression until you can explain why your location for the conversion is the better design.

In [13]:
# DEBUG ME
portal_response = {'submitted_applications': '12'}  # External API payload: strings are untrusted.
referral_applications: int = 6
portal_applications = int(portal_response['submitted_applications'])
total_applications = portal_applications + referral_applications

print(f'Total applications: {total_applications}')

Total applications: 18


## Your turn 2 — write a small parser

Implement `parse_nonnegative_hours(raw_hours: str) -> float`. It must convert valid text such as `'7.5'`, reject non-numeric and negative values with clear `ValueError`s, and return a `float` including for `'8'`.

Use the test cell only after implementing it. Read a failure as feedback, not as a reason to change the test.

In [ ]:
# YOUR TURN
def parse_nonnegative_hours(raw_hours: str) -> float:
    # Replace this placeholder with your implementation.
    try:
        numeric_value=float(raw_hours.strip())
    except ValueError as error:
        raise ValueError(f"not numeric value{raw_hours!r}") from error
    if numeric_value < 0:
        raise ValueError (f"hours can't be negative{numeric_value!r}")
    return numeric_value
   


In [15]:
# Tests — run after completing parse_nonnegative_hours.
assert parse_nonnegative_hours('7.5') == 7.5
assert parse_nonnegative_hours('8') == 8.0

for invalid_hours in ('not-a-number', '-0.5'):
    try:
        parse_nonnegative_hours(invalid_hours)
    except ValueError:
        pass
    else:
        raise AssertionError(f'{invalid_hours!r} should have raised ValueError')

print('All parser checks passed.')

All parser checks passed.


## Mini interview exercise — normalize an API payload

Implement `normalize_candidate`. It accepts the raw payload below and returns a new dictionary with this contract:

```python
{'user_id': int, 'username': str, 'verified': bool, 'score': float}
```

Use `parse_boolean` for the boolean. Raise `ValueError` with a field-specific message if a conversion fails. Do not mutate the input payload.

In [ ]:
# YOUR TURN
def normalize_candidate(raw_candidate: Mapping[str, str]) -> dict[str, int | str | bool | float]:
    # Implement the contract described above.
    try:
        user_id=int(raw_candidate["user_id"])
    except (KeyError,ValueError) as error:
        raise ValueError("Invalid user id:") from error
    try:
        username=(raw_candidate["username"])
    except (KeyError) as error:
        raise ValueError("Misssing username") from error
                        
    try:
        verified = parse_boolean(raw_candidate["verified"]) # Call your existing parse_boolean function here.
    except (KeyError, ValueError) as error:
        raise ValueError("Invalid verified value: ...") from error

    try:
        score = float(raw_candidate["score"])  # Read "score" and convert it to float.
    except (KeyError, ValueError) as error:
        raise ValueError("Invalid score: ") from error

    return {
        "user_id": user_id,
        "username": username,
        "verified": verified,
        "score": score,
    }

raw_candidate = {
    'user_id': '42',
    'username': 'sam',
    'verified': 'TRUE',
    'score': '91.5',
}

normalized_candidate = normalize_candidate(raw_candidate)
print(normalized_candidate)
print({key: type(value).__name__ for key, value in normalized_candidate.items()})

{'user_id': 42, 'username': 'sam', 'verified': True, 'score': 91.5}
{'user_id': 'int', 'username': 'str', 'verified': 'bool', 'score': 'float'}


## Explain it like an engineer

Answer these without running code:

1. Why should parsing happen at a system boundary?
2. What is the difference between `bool('false')` and `parse_boolean('false')`?
3. Why are `ValueError` messages part of production-quality code?
4. What do the `assert` statements prove—and what do they not prove?
5. What happens if `int('12.5')` is called, and how would you explain the error?

When you finish, send me your fixed debugging cell and the two functions. I’ll code-review them, point out trade-offs, and only then create the next notebook on conditions and loops.